# DROP TARGET DB
#### Duplicate a source DB

In [22]:
import pymongo

def duplicate_db(source, target, uri="mongodb://localhost:27017/"):
    mongo_client = pymongo.MongoClient(uri)
    source_db = mongo_client[source]
    target_db = mongo_client[target]

    # Clear the Database if it Exists
    mongo_client.drop_database(target)


    for coll_name in source_db.list_collection_names():
        source_coll = source_db[coll_name]
        target_coll = target_db[coll_name]


        docs = list(source_coll.find({}))
        if docs:
            target_coll.insert_many(docs)
        print(f"Copied {len(docs)} documents from {coll_name}")

    print(f"Database '{source}' successfully duplicated to '{target}'")

duplicate_db("ceur_ws_test", "ceur_ws_etl")

Copied 139031 documents from all_papers
Copied 488199 documents from authors
Copied 817 documents from volumes
Copied 127364 documents from related_papers
Copied 11796 documents from papers
Copied 288855 documents from authors_grouped
Database 'ceur_ws_test' successfully duplicated to 'ceur_ws_etl'


## Connection

In [23]:
from bson import ObjectId

client = pymongo.MongoClient("mongodb://localhost:27017/")

db = client['ceur_ws_etl']


volumes = db['volumes']

papers = db['papers']
related_papers = db['related_papers']
all_papers = db['all_papers']

authors = db['authors']
authors_grouped = db['authors_grouped']

keywords = db['keywords']
keywords_final = db['keywords_final']

abstracts = db['abstracts']

# Create Collections

Related Papers Collection + Associate each Related Paper with an ID and to the Parent Paper.


In [24]:
# Clear the Collection if it Exists
related_papers.delete_many({})


pipeline = [
    {"$unwind": "$paper_info.related_papers"},
    {"$match": {
        "$expr": {
            "$gte": [
                {"$strLenCP": "$paper_info.related_papers.title"},
                20
            ]
        }
    }},
    {"$group": {
        "_id": "$paper_info.related_papers.title",
        "count": {"$sum": 1},
        "paper_ids": {"$push": "$_id"},
        "authors_set": {"$addToSet": "$paper_info.related_papers.authors"},
        "texts": {"$addToSet": "$paper_info.related_papers.text"}
    }},
    {"$sort": {"count": -1}},
    {"$addFields": {
        "cleaned_authors_sets": {
            "$map": {
                "input": "$authors_set",
                "as": "authors",
                "in": {
                    "$filter": {
                        "input": "$$authors",
                        "as": "a",
                        "cond": {
                            "$and": [
                                {"$gt": [{"$strLenCP": {"$trim": {"input": "$$a"}}}, 2]},
                                {"$not": [{"$in": [{"$toLower": "$$a"}, ["et al.", "et al"]]}]},
                                {"$not": [{"$regexMatch": {"input": "$$a", "regex": "BERT", "options": "i"}}]}
                            ]
                        }
                    }
                }
            }
        }
    }},
    {"$addFields": {
        "best_authors": {
            "$reduce": {
                "input": "$cleaned_authors_sets",
                "initialValue": [],
                "in": {
                    "$cond": [
                        {"$gt": [{"$size": "$$this"}, {"$size": "$$value"}]},
                        "$$this",
                        "$$value"
                    ]
                }
            }
        }
    }},
]

results = list(papers.aggregate(pipeline))

for doc in results:
    doc['title'] = doc.pop('_id')
    doc['_id'] = ObjectId()
    doc['text'] = doc['texts'][0] if doc['texts'] else None
    doc.pop('texts', None)
    doc['cleaned_authors_set'] = doc.pop('cleaned_authors_sets', None)
    related_papers.insert_one(doc)

Merge the Related Paper and Papers Collection into an All Papers Collection.

Since the Paper has more Information, the Related Paper is Merged into the Paper Collection.

In [25]:
# Clear the Collection if it Exists
all_papers.delete_many({})
title_to_id = dict()


for paper in papers.find():
    title_to_id[paper['title']] = paper['_id']
    paper['paper_id'] = paper['_id']
    paper['_id'] = ObjectId()
    paper['from'] = 'paper'
    paper['count'] = 1
    all_papers.insert_one(paper)


merged = 0
for rpaper in related_papers.find():
    if rpaper['title'] in title_to_id:
        merged += 1
        all_papers.update_one(
            {'_id': title_to_id[rpaper['title']]},
            {'$inc': {'count': rpaper['count']}}
        )
    else:
        rpaper['paper_id'] = rpaper['_id']
        rpaper['_id'] = ObjectId()
        rpaper['from'] = 'related'
        all_papers.insert_one(rpaper)

print(f"Merged {merged} related papers into existing papers.")

Merged 129 related papers into existing papers.


Authors Collection from the Related Papers and Papers Collections.

In [26]:
# Clear the Collection if it Exists
authors.delete_many({})

pipeline_related = [
    {"$unwind": "$best_authors"},
    {"$project": {
        "name": "$best_authors",
        "from": {"$literal": "related"},
        "related_id": "$_id"
    }}
]

pipeline_papers = [
    {"$unwind": "$author"},
    {"$project": {
        "name": "$author",
        "from": {"$literal": "paper"},
        "paper_id": "$_id"
    }}
]

pipeline = pipeline_related + [
    {"$unionWith": {
        "coll": "papers",
        "pipeline": pipeline_papers
    }},
    {"$addFields": {"_id": {"$function": {
        "body": "function() { return new ObjectId(); }",
        "args": [],
        "lang": "js"
    }}}},
    {"$out": "authors"}
]

related_papers.aggregate(pipeline)


# Clear the Collection if it Exists
authors_grouped.delete_many({})

pipeline = [
    {
        "$addFields": {
            "name_trimmed": {"$trim": {"input": "$name"}}
        }
    },
    {
        "$match": {
            "name_trimmed": {
                "$ne": None,
                "$not": {"$regex": r"\d"}
            }
        }
    },
    {
        "$group": {
            "_id": "$name_trimmed",
            "ids": {"$push": "$_id"},
            "from_set": {"$addToSet": "$from"},
            "paper_ids": {"$addToSet": "$paper_id"},
            "related_ids": {"$addToSet": "$related_id"}
        }
    },
    {
        "$out": "authors_grouped"
    }
]
authors.aggregate(pipeline)
authors_grouped.find_one()['_id']


'" A novel verilog-A'

Keywords Collection from Papers Collection

In [27]:
pipeline = [
    {
        "$unwind": "$paper_info.keywords"
    },
    {
        "$project": {
            "name": "$paper_info.keywords",
            "paper_id": "$_id"
        }
    },
    {
        "$group": {
            "_id": "$name",
            "paper_ids": {"$addToSet": "$paper_id"}
        }
    },
    {
        "$project": {
            "_id": 0,
            "name": "$_id",
            "paper_ids": 1
        }
    },
    {
        "$merge": {
            "into": "keywords_final",
            "whenMatched": "replace",
            "whenNotMatched": "insert"
        }
    }
]

papers.aggregate(pipeline)

Abstract Collection from Papers Collection.


In [28]:
# Clear the Collection if it Exists
abstracts.delete_many({})

pipeline_abstract = [
    {
        "$match": {
            "abstract": {
                "$ne": None,
                "$not": {"$in": [""]},
               # "$expr": {"$gte": [{"$strLenCP": "$abstract"}, 50]}
            }
        }
    },
    {
        "$project": {
            "_id": 0,
            "text": "$abstract",
            "paper_id": "$_id"
        }
    },
    {
        "$addFields": {
            "_id": {"$function": {
                "body": "function() { return new ObjectId(); }",
                "args": [],
                "lang": "js"
            }}
        }
    },
    {
        "$out": "abstracts"
    }
]

papers.aggregate(pipeline_abstract)

abstracts.find_one()["text"]

"Computational thinking and educational robotics are becoming key competencies for creating competent digital citizens in today's world. The development of these skills has been gradually implemented in Primary and Secondary Education, but there is still a long way to go, especially in their use in Early Childhood Education. The use of these technologies from an early age has shown to have positive effects on students' education. This paper presents an intervention among 3-year-old students using the Bee-Bot robot. The study includes both unplugged activities and activities with the robot to develop computational thinking skills. The results show an improvement in the acquisition of these concepts with meaningful learning after conducting the robotics sessions. Additionally, the obtained results are analysed and options for their improvement are discussed. The difficulties and limitations of this study are also addressed."

# Create Memgraph

### Connection

In [68]:
from gqlalchemy import Memgraph

host_memgraph = "127.0.0.1"
port_memgraph = 7685
memgraph = Memgraph(host=host_memgraph, port=port_memgraph)


# Clear the Database
query_delete = """
    MATCH (n)
    DETACH DELETE n
"""

memgraph.execute(query_delete)

print("Database Cleared.")

def execute_batch(collection, query, batch_size = 10_000):
    batch = []
    batch_counter = 0
    for item in collection:
        item['mongo_id'] = str(item.pop('_id'))

        batch.append(item)

        if len(batch) >= batch_size:
            batch_counter += 1
            memgraph.execute(query, {"batch": batch})
            print(f"Inserted Batch {batch_counter} ({batch_size} Nodes)")

            batch = []

    if batch:
        batch_counter += 1
        memgraph.execute(query, {"batch": batch})
        print(f"Inserted Final Batch {batch_counter} ({len(batch)} items)")

Database Cleared.


### Nodes

In [69]:
## ADD Papers

all_papers_select = all_papers.find({}, {
    '_id': "$paper_id",
    'title': 1,
    'from': 1,
    'count': 1,
    'text': 1
})

query_add_papers = """
    UNWIND $batch AS row
    CREATE (:Paper {
        id: row.mongo_id,
        name: row.title,
        source: row.from,
        count: row.count,
        text: row.text
    })
"""

execute_batch(all_papers_select, query_add_papers)


Inserted Batch 1 (10000 Nodes)
Inserted Batch 2 (10000 Nodes)
Inserted Batch 3 (10000 Nodes)
Inserted Batch 4 (10000 Nodes)
Inserted Batch 5 (10000 Nodes)
Inserted Batch 6 (10000 Nodes)
Inserted Batch 7 (10000 Nodes)
Inserted Batch 8 (10000 Nodes)
Inserted Batch 9 (10000 Nodes)
Inserted Batch 10 (10000 Nodes)
Inserted Batch 11 (10000 Nodes)
Inserted Batch 12 (10000 Nodes)
Inserted Batch 13 (10000 Nodes)
Inserted Final Batch 14 (9031 items)


In [70]:
## ADD Authors

authors_select = authors_grouped.find({}, {
    '_id': 1,
})

query_add_authors = """
    UNWIND $batch as row
    CREATE(:Author {
        name: row.mongo_id
    })
"""

execute_batch(authors_select, query_add_authors)

Inserted Batch 1 (10000 Nodes)
Inserted Batch 2 (10000 Nodes)
Inserted Batch 3 (10000 Nodes)
Inserted Batch 4 (10000 Nodes)
Inserted Batch 5 (10000 Nodes)
Inserted Batch 6 (10000 Nodes)
Inserted Batch 7 (10000 Nodes)
Inserted Batch 8 (10000 Nodes)
Inserted Batch 9 (10000 Nodes)
Inserted Batch 10 (10000 Nodes)
Inserted Batch 11 (10000 Nodes)
Inserted Batch 12 (10000 Nodes)
Inserted Batch 13 (10000 Nodes)
Inserted Batch 14 (10000 Nodes)
Inserted Batch 15 (10000 Nodes)
Inserted Batch 16 (10000 Nodes)
Inserted Batch 17 (10000 Nodes)
Inserted Batch 18 (10000 Nodes)
Inserted Batch 19 (10000 Nodes)
Inserted Batch 20 (10000 Nodes)
Inserted Batch 21 (10000 Nodes)
Inserted Batch 22 (10000 Nodes)
Inserted Batch 23 (10000 Nodes)
Inserted Batch 24 (10000 Nodes)
Inserted Batch 25 (10000 Nodes)
Inserted Batch 26 (10000 Nodes)
Inserted Batch 27 (10000 Nodes)
Inserted Final Batch 28 (7623 items)


In [71]:
## ADD Volumes

volumes_select = volumes.find({}, {
    '_id': 1,
    'title': 1,
    'pubyear': 1
})

query_add_volumes = """
    UNWIND $batch as row
    CREATE (:Volume {
        id: row.mongo_id,
        name: row.title,
        year: row.pubyear
    })
"""

execute_batch(volumes_select, query_add_volumes)

Inserted Final Batch 1 (817 items)


In [72]:
## ADD Keywords

keywords_select = keywords_final.find({}, {
    '_id': 1,
    'title': 1,

})

query_add_keywords = """
    UNWIND $batch as row
    CREATE (:Keyword {
        id: row.mongo_id,
        name: row.title
    })
"""

execute_batch(keywords_select, query_add_keywords)

Inserted Batch 1 (10000 Nodes)
Inserted Batch 2 (10000 Nodes)
Inserted Batch 3 (10000 Nodes)
Inserted Final Batch 4 (5439 items)


In [73]:
## ADD Abstracts

abstracts_select = abstracts.find({}, {
    '_id': 1,
    'text': 1,

})

query_add_abstracts = """
    UNWIND $batch as row
    CREATE (:Abstract {
        id: row.mongo_id,
        text: row.text
    })
"""

execute_batch(abstracts_select, query_add_abstracts)

Inserted Final Batch 1 (9695 items)


In [74]:
# ADD INDEXES
memgraph.execute("CREATE INDEX ON :Paper(id)")
memgraph.execute("CREATE INDEX ON :Author(name)")
memgraph.execute("CREATE INDEX ON :Volume(id)")
memgraph.execute("CREATE INDEX ON :Keyword(id)")
memgraph.execute("CREATE INDEX ON :Abstract(id)")

### Relationships

In [75]:
## Paper - Author
query_paper_author = """
    UNWIND $batch AS row
    MATCH (a:Author {name: row.mongo_id})
    MATCH (p:Paper {id: row.target_id})
    MERGE (a)-[:WROTE]->(p);
"""
pipeline = [
    {
        "$project": {
            "ids": 1,
            "paper_ids": 1,
            "related_ids": 1
        }
    },
    {
        "$unwind": {
            "path": "$paper_ids",
            "preserveNullAndEmptyArrays": True
        }
    },
    {
        "$unwind": {
            "path": "$related_ids",
            "preserveNullAndEmptyArrays": True
        }
    },
    {
        "$project": {
            "_id": 1,
            "paper_id": "$paper_ids",
            "related_id": "$related_ids"
        }
    },
    {
        "$project": {
            "_id": 1,
            "target_id": {
                "$toString": {"$ifNull": ["$paper_id", "$related_id"]}
            }
        }
    },
    {
        "$match": {
            "target_id": {"$ne": None}
        }
    }
]
print(authors_grouped.aggregate(pipeline).next())
execute_batch(authors_grouped.aggregate(pipeline), query_paper_author)




{'_id': '" A novel verilog-A', 'target_id': '68e3e81656fce2f2c5557876'}
Inserted Batch 1 (10000 Nodes)
Inserted Batch 2 (10000 Nodes)
Inserted Batch 3 (10000 Nodes)
Inserted Batch 4 (10000 Nodes)
Inserted Batch 5 (10000 Nodes)
Inserted Batch 6 (10000 Nodes)
Inserted Batch 7 (10000 Nodes)
Inserted Batch 8 (10000 Nodes)
Inserted Batch 9 (10000 Nodes)
Inserted Batch 10 (10000 Nodes)
Inserted Batch 11 (10000 Nodes)
Inserted Batch 12 (10000 Nodes)
Inserted Batch 13 (10000 Nodes)
Inserted Batch 14 (10000 Nodes)
Inserted Batch 15 (10000 Nodes)
Inserted Batch 16 (10000 Nodes)
Inserted Batch 17 (10000 Nodes)
Inserted Batch 18 (10000 Nodes)
Inserted Batch 19 (10000 Nodes)
Inserted Batch 20 (10000 Nodes)
Inserted Batch 21 (10000 Nodes)
Inserted Batch 22 (10000 Nodes)
Inserted Batch 23 (10000 Nodes)
Inserted Batch 24 (10000 Nodes)
Inserted Batch 25 (10000 Nodes)
Inserted Batch 26 (10000 Nodes)
Inserted Batch 27 (10000 Nodes)
Inserted Batch 28 (10000 Nodes)
Inserted Batch 29 (10000 Nodes)
Inserted 